In [1]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path.cwd()

FILES = {
    'customers_dataset': 'customers_dataset.csv',
    'geolocation_dataset': 'geolocation_dataset.csv',
    'order_items_dataset': 'order_items_dataset.csv',
    'order_payments_dataset': 'order_payments_dataset.csv',
    'order_reviews_dataset': 'order_reviews_dataset.csv',
    'orders_dataset': 'orders_dataset.csv',
    'product_category_name_translation': 'product_category_name_translation.csv',
    'products_dataset': 'products_dataset.csv',
    'sellers_dataset': 'sellers_dataset.csv',
}

EXPECTED_COLUMNS = {
    'customers_dataset': {
        'customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state'
    },
    'geolocation_dataset': {
        'geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state'
    },
    'order_items_dataset': {
        'order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value'
    },
    'order_payments_dataset': {
        'order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value'
    },
    'order_reviews_dataset': {
        'review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message',
        'review_creation_date', 'review_answer_timestamp'
    },
    'orders_dataset': {
        'order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at',
        'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date'
    },
    'product_category_name_translation': {
        'product_category_name', 'product_category_name_english'
    },
    'products_dataset': {
        'product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght',
        'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm'
    },
    'sellers_dataset': {
        'seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state'
    },
}

dataframes = {}
load_report = []

In [2]:
for dataset_name, filename in FILES.items():
    file_path = BASE_DIR / filename
    record = {
        'dataset': dataset_name,
        'file': filename,
        'exists': file_path.exists(),
        'loaded': False,
        'rows': None,
        'cols': None,
        'missing_expected_columns': None,
        'error': None,
    }

    if not file_path.exists():
        record['error'] = 'Archivo no encontrado'
        load_report.append(record)
        continue

    try:
        df = pd.read_csv(file_path)
        dataframes[dataset_name] = df
        missing_cols = sorted(list(EXPECTED_COLUMNS[dataset_name] - set(df.columns)))

        record['loaded'] = True
        record['rows'] = int(df.shape[0])
        record['cols'] = int(df.shape[1])
        record['missing_expected_columns'] = ', '.join(missing_cols) if missing_cols else ''

    except Exception as exc:
        record['error'] = str(exc)

    load_report.append(record)

load_report_df = pd.DataFrame(load_report)
load_report_df

,dataset,file,exists,loaded,rows,cols,missing_expected_columns,error
0,customers_dataset,customers_dataset.csv,True,True,99441,5,,None
1,geolocation_dataset,geolocation_dataset.csv,True,True,1000163,5,,None
2,order_items_dataset,order_items_dataset.csv,True,True,112650,7,,None
3,order_payments_dataset,order_payments_dataset.csv,True,True,103886,5,,None
4,order_reviews_dataset,order_reviews_dataset.csv,True,True,99224,7,,None
5,orders_dataset,orders_dataset.csv,True,True,99441,8,,None
6,product_category_name_translation,product_category_name_translation.csv,True,True,71,2,,None
7,products_dataset,products_dataset.csv,True,True,32951,9,,None
8,sellers_dataset,sellers_dataset.csv,True,True,3095,4,,None


In [3]:
quality_checks = []

def add_check(check, passed, details=''):
    quality_checks.append({'check': check, 'passed': bool(passed), 'details': details})

# 1) Archivos cargados y no vacíos
for dataset_name, filename in FILES.items():
    if dataset_name not in dataframes:
        add_check(f'{dataset_name} cargado', False, 'No se pudo cargar el archivo')
        continue

    df = dataframes[dataset_name]
    add_check(f'{dataset_name} no vacío', df.shape[0] > 0, f'rows={df.shape[0]}')

    missing_cols = EXPECTED_COLUMNS[dataset_name] - set(df.columns)
    add_check(
        f'{dataset_name} columnas esperadas',
        len(missing_cols) == 0,
        '' if not missing_cols else f'Missing: {sorted(missing_cols)}'
    )

# 2) Duplicados en llaves principales
if 'customers_dataset' in dataframes:
    dup = dataframes['customers_dataset'].duplicated(subset=['customer_id']).sum()
    add_check('customer_id único', dup == 0, f'duplicados={int(dup)}')

if 'orders_dataset' in dataframes:
    dup = dataframes['orders_dataset'].duplicated(subset=['order_id']).sum()
    add_check('order_id único', dup == 0, f'duplicados={int(dup)}')

if 'products_dataset' in dataframes:
    dup = dataframes['products_dataset'].duplicated(subset=['product_id']).sum()
    add_check('product_id único', dup == 0, f'duplicados={int(dup)}')

if 'sellers_dataset' in dataframes:
    dup = dataframes['sellers_dataset'].duplicated(subset=['seller_id']).sum()
    add_check('seller_id único', dup == 0, f'duplicados={int(dup)}')

if 'order_items_dataset' in dataframes:
    dup = dataframes['order_items_dataset'].duplicated(subset=['order_id', 'order_item_id']).sum()
    add_check('order_items (order_id, order_item_id) único', dup == 0, f'duplicados={int(dup)}')

if 'order_payments_dataset' in dataframes:
    dup = dataframes['order_payments_dataset'].duplicated(subset=['order_id', 'payment_sequential']).sum()
    add_check('order_payments (order_id, payment_sequential) único', dup == 0, f'duplicados={int(dup)}')

# 3) Integridad referencial
if 'orders_dataset' in dataframes and 'customers_dataset' in dataframes:
    child = set(dataframes['orders_dataset']['customer_id'].dropna().astype(str).unique())
    parent = set(dataframes['customers_dataset']['customer_id'].dropna().astype(str).unique())
    missing = child - parent
    add_check('orders.customer_id existe en customers', len(missing) == 0, f'missing_refs={len(missing)}')

if 'order_items_dataset' in dataframes and 'orders_dataset' in dataframes:
    child = set(dataframes['order_items_dataset']['order_id'].dropna().astype(str).unique())
    parent = set(dataframes['orders_dataset']['order_id'].dropna().astype(str).unique())
    missing = child - parent
    add_check('order_items.order_id existe en orders', len(missing) == 0, f'missing_refs={len(missing)}')

if 'order_items_dataset' in dataframes and 'products_dataset' in dataframes:
    child = set(dataframes['order_items_dataset']['product_id'].dropna().astype(str).unique())
    parent = set(dataframes['products_dataset']['product_id'].dropna().astype(str).unique())
    missing = child - parent
    add_check('order_items.product_id existe en products', len(missing) == 0, f'missing_refs={len(missing)}')

if 'order_items_dataset' in dataframes and 'sellers_dataset' in dataframes:
    child = set(dataframes['order_items_dataset']['seller_id'].dropna().astype(str).unique())
    parent = set(dataframes['sellers_dataset']['seller_id'].dropna().astype(str).unique())
    missing = child - parent
    add_check('order_items.seller_id existe en sellers', len(missing) == 0, f'missing_refs={len(missing)}')

if 'order_payments_dataset' in dataframes and 'orders_dataset' in dataframes:
    child = set(dataframes['order_payments_dataset']['order_id'].dropna().astype(str).unique())
    parent = set(dataframes['orders_dataset']['order_id'].dropna().astype(str).unique())
    missing = child - parent
    add_check('order_payments.order_id existe en orders', len(missing) == 0, f'missing_refs={len(missing)}')

if 'order_reviews_dataset' in dataframes and 'orders_dataset' in dataframes:
    child = set(dataframes['order_reviews_dataset']['order_id'].dropna().astype(str).unique())
    parent = set(dataframes['orders_dataset']['order_id'].dropna().astype(str).unique())
    missing = child - parent
    add_check('order_reviews.order_id existe en orders', len(missing) == 0, f'missing_refs={len(missing)}')

if 'products_dataset' in dataframes and 'product_category_name_translation' in dataframes:
    child = set(dataframes['products_dataset']['product_category_name'].dropna().astype(str).unique())
    parent = set(dataframes['product_category_name_translation']['product_category_name'].dropna().astype(str).unique())
    missing = child - parent
    add_check('products.product_category_name existe en translation', len(missing) == 0, f'missing_refs={len(missing)}')

quality_df = pd.DataFrame(quality_checks)
quality_df

,check,passed,details
0,customers_dataset no vacío,True,rows=99441
1,customers_dataset columnas esperadas,True,
2,geolocation_dataset no vacío,True,rows=1000163
3,geolocation_dataset columnas esperadas,True,
4,order_items_dataset no vacío,True,rows=112650
5,order_items_dataset columnas esperadas,True,
6,order_payments_dataset no vacío,True,rows=103886
7,order_payments_dataset columnas esperadas,True,
8,order_reviews_dataset no vacío,True,rows=99224
9,order_reviews_dataset columnas esperadas,True,


In [4]:
# Resumen de nulos por dataset (top 5 columnas con más nulos)
nulls_summary = []

for dataset_name, df in dataframes.items():
    total_rows = len(df)
    if total_rows == 0:
        continue

    null_counts = df.isna().sum().sort_values(ascending=False)
    top_nulls = null_counts.head(5)

    for col, count in top_nulls.items():
        pct = (count / total_rows) * 100
        nulls_summary.append({
            'dataset': dataset_name,
            'column': col,
            'null_count': int(count),
            'null_pct': round(float(pct), 2),
        })

nulls_df = pd.DataFrame(nulls_summary).sort_values(['dataset', 'null_count'], ascending=[True, False])
nulls_df.head(30)

,dataset,column,null_count,null_pct
0,customers_dataset,customer_id,0,0.00
1,customers_dataset,customer_unique_id,0,0.00
2,customers_dataset,customer_zip_code_prefix,0,0.00
3,customers_dataset,customer_city,0,0.00
4,customers_dataset,customer_state,0,0.00
5,geolocation_dataset,geolocation_zip_code_prefix,0,0.00
6,geolocation_dataset,geolocation_lat,0,0.00
7,geolocation_dataset,geolocation_lng,0,0.00
8,geolocation_dataset,geolocation_city,0,0.00
9,geolocation_dataset,geolocation_state,0,0.00


In [5]:
total_checks = len(quality_df)
passed_checks = int(quality_df['passed'].sum()) if total_checks else 0
failed_checks = total_checks - passed_checks

print('=== RESULTADO DE VALIDACIÓN ===')
print(f'Checks totales: {total_checks}')
print(f'Checks aprobados: {passed_checks}')
print(f'Checks fallidos: {failed_checks}')

if failed_checks == 0:
    print('Estado general: ✅ Todos los datasets pasaron las validaciones implementadas.')
else:
    print('Estado general: ⚠️ Hay validaciones fallidas, revisar detalle abajo.')
    display(quality_df[~quality_df['passed']])

=== RESULTADO DE VALIDACIÓN ===
Checks totales: 31
Checks aprobados: 30
Checks fallidos: 1
Estado general: ⚠️ Hay validaciones fallidas, revisar detalle abajo.


,check,passed,details
30,products.product_category_name existe en trans...,False,missing_refs=2


In [6]:
products_df = dataframes['products_dataset'].copy()
translation_df = dataframes['product_category_name_translation'].copy()

products_categories = set(products_df['product_category_name'].dropna().astype(str).str.strip().unique())
translated_categories = set(translation_df['product_category_name'].dropna().astype(str).str.strip().unique())

missing_categories = sorted(products_categories - translated_categories)

print(f'Categorías faltantes en translation: {len(missing_categories)}')

if missing_categories:
    missing_df = pd.DataFrame({'missing_product_category_name': missing_categories})
    display(missing_df)
else:
    print('No hay categorías faltantes.')

Categorías faltantes en translation: 2


,missing_product_category_name
0,pc_gamer
1,portateis_cozinha_e_preparadores_de_alimentos
